In [1]:
"""
Fashion-MNIST MLP baseline: 784 -> 512 -> 512 -> 10, ReLU.
Trains an FP32 reference model and saves the weights.
This checkpoint is meant to be frozen -- downstream experiments load it, never retrain it.
"""
 
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
# ----------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------
SEED        = 0
EPOCHS      = 40
BATCH_SIZE  = 128
LR          = 1e-3
DATA_DIR    = "./data"
CKPT_PATH   = "fmnist_mlp_fp32.pt"
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
# Reproducibility -- matters here because the checkpoint is a fixed artifact
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
# ----------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------
# Fashion-MNIST channel stats (computed over the training split)
MEAN, STD = 0.2860, 0.3530
 
tfm = transforms.Compose([
    transforms.ToTensor(),                 # [0,1], shape (1,28,28)
    transforms.Normalize((MEAN,), (STD,)),
    transforms.Lambda(lambda x: x.view(-1)),  # flatten -> (784,)
])
 
train_set = datasets.FashionMNIST(DATA_DIR, train=True,  download=True, transform=tfm)
test_set  = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=tfm)
 
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=False)
test_loader  = DataLoader(test_set,  batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)
 
CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 166kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.18MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 18.9MB/s]


In [4]:
# ----------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=512, n_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, n_classes)
 
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)          # logits
 
 
model = MLP().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
 
n_params = sum(p.numel() for p in model.parameters())
print(f"device: {DEVICE} | params: {n_params:,} | fp32 size: {n_params * 4 / 1e6:.2f} MB")

device: cuda | params: 669,706 | fp32 size: 2.68 MB


In [5]:
# ----------------------------------------------------------------------
# Train / eval loops
# ----------------------------------------------------------------------
@torch.no_grad()
def evaluate(loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss_sum += criterion(logits, y).item() * y.size(0)
        correct  += (logits.argmax(1) == y).sum().item()
        total    += y.size(0)
    return loss_sum / total, correct / total
 
 
def train_one_epoch():
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()
 
        loss_sum += loss.item() * y.size(0)
        correct  += (logits.argmax(1) == y).sum().item()
        total    += y.size(0)
    return loss_sum / total, correct / total
 
 
history = []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch()
    te_loss, te_acc = evaluate(test_loader)
    history.append((epoch, tr_loss, tr_acc, te_loss, te_acc))
    print(f"epoch {epoch:3d}/{EPOCHS} | "
          f"train loss {tr_loss:.4f} acc {tr_acc*100:.2f}% | "
          f"test loss {te_loss:.4f} acc {te_acc*100:.2f}%")
 
print(f"\ntotal time: {time.time() - t0:.1f}s")

epoch   1/40 | train loss 0.4621 acc 83.12% | test loss 0.4031 acc 85.25%
epoch   2/40 | train loss 0.3402 acc 87.45% | test loss 0.3776 acc 86.21%
epoch   3/40 | train loss 0.3014 acc 88.69% | test loss 0.3457 acc 87.46%
epoch   4/40 | train loss 0.2770 acc 89.53% | test loss 0.3282 acc 88.09%
epoch   5/40 | train loss 0.2566 acc 90.33% | test loss 0.3366 acc 88.13%
epoch   6/40 | train loss 0.2404 acc 90.95% | test loss 0.3260 acc 88.15%
epoch   7/40 | train loss 0.2243 acc 91.46% | test loss 0.3169 acc 88.66%
epoch   8/40 | train loss 0.2089 acc 92.01% | test loss 0.3429 acc 88.13%
epoch   9/40 | train loss 0.1971 acc 92.36% | test loss 0.3519 acc 88.46%
epoch  10/40 | train loss 0.1877 acc 92.68% | test loss 0.3332 acc 89.21%
epoch  11/40 | train loss 0.1717 acc 93.44% | test loss 0.3519 acc 89.32%
epoch  12/40 | train loss 0.1648 acc 93.65% | test loss 0.3574 acc 89.02%
epoch  13/40 | train loss 0.1578 acc 94.03% | test loss 0.3681 acc 88.99%
epoch  14/40 | train loss 0.1476 acc 9

In [6]:
# ----------------------------------------------------------------------
# Save the frozen FP32 reference
# ----------------------------------------------------------------------
final_loss, final_acc = evaluate(test_loader)
 
torch.save({
    "state_dict":  model.state_dict(),          # all tensors are fp32
    "arch":        {"in_dim": 784, "hidden": 512, "n_classes": 10},
    "normalize":   {"mean": MEAN, "std": STD},
    "test_acc":    final_acc,
    "epochs":      EPOCHS,
    "seed":        SEED,
    "history":     history,
}, CKPT_PATH)
 
print(f"saved -> {CKPT_PATH}  ({os.path.getsize(CKPT_PATH)/1e6:.2f} MB)  "
      f"final test acc {final_acc*100:.2f}%")

saved -> fmnist_mlp_fp32.pt  (2.68 MB)  final test acc 89.89%


In [7]:
# ----------------------------------------------------------------------
# Reload sanity check -- confirms the checkpoint reproduces the accuracy
# ----------------------------------------------------------------------
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
ref = MLP(**ckpt["arch"]).to(DEVICE)
ref.load_state_dict(ckpt["state_dict"])
ref.eval()
 
model = ref  # so evaluate() uses the reloaded copy
_, reloaded_acc = evaluate(test_loader)
assert abs(reloaded_acc - final_acc) < 1e-9, "checkpoint did not reload cleanly"
print(f"reload check OK: {reloaded_acc*100:.2f}%")

reload check OK: 89.89%
